# ЛР 3: анализ результатов
Выполни все ячейки. API не вызывается. A/B/E проверяются автоматически; C/D — двумя проверяющими, B/F — по рубрике. Заполненные ИИ оценки предварительные и не заменяют независимую человеческую проверку. Пустая оценка означает «не проверено».
Открывай тетрадь из корня проекта. Журналы лежат в `logs/`, таблицы и графики сохраняются в `results/`.

In [ ]:
import hashlib, itertools, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

FOLDER = Path.cwd()
OUT = FOLDER / 'results'
OUT.mkdir(exist_ok=True)
MODELS = ['gigachat', 'gpt', 'local']
questions = pd.read_csv(FOLDER / 'data' / 'questions.csv', dtype=str).fillna('').set_index('id')
records = [json.loads(s) for filename in ['logs/api_responses.jsonl', 'logs/local_responses.jsonl', 'logs/colab_experiments.jsonl']
           if (path := FOLDER / filename).exists() for s in path.read_text().splitlines()]
if any(r.get('model') == 'tlite' and r.get('id') in questions.index for r in records):
    MODELS.append('tlite')
selected = {}
for r in records:
    key = (r.get('id'), r.get('model'))
    if key[0] not in questions.index or key[1] not in MODELS:
        continue
    previous = selected.get(key, {})
    if previous.get('answer') and not previous.get('error') and previous.get('finish_reason') != 'length':
        continue
    selected[key] = r
rows = []
for qid, model in itertools.product(questions.index, MODELS):
    r = selected.get((qid, model), {})
    error = r.get('error', '')
    if r and ('answer' not in r or r.get('finish_reason') == 'length'):
        error = error or 'Нет полного ответа'
    rows.append({'id': qid, 'model': model, **questions.loc[qid].to_dict(),
                 'answer': r.get('answer', ''), 'error': error, 'received': bool(r) and not error,
                 'seconds': r.get('seconds'), 'tokens': r.get('tokens'), 'date': r.get('date', ''),
                 'prompt_tokens': r.get('prompt_tokens'), 'completion_tokens': r.get('completion_tokens'), 'peak_gpu_gib': r.get('peak_gpu_gib'),
                 'gpu': r.get('gpu', '')})
df = pd.DataFrame(rows)

def save(table, filename):
    flat = table.map(lambda v: v.replace('\r\n', '\n').replace('\r', '\n').replace('\n', r'\n') if isinstance(v, str) else v)
    flat.to_csv(OUT / filename, index=False, encoding='utf-8-sig')

## Проверка ответов
Для чисел используется допуск из датасета; проценты и десятичная запятая нормализуются. В A03 проверяются ширина и класс. В B автоматически проверяется метка, обоснование оценивается отдельно.

In [ ]:
def numbers(text):
    text = re.sub(r'\\(?:d?frac)\s*\{([-+\d.,]+)\}\s*\{([-+\d.,]+)\}', r'\1/\2', text)
    tokens = re.findall(r'[-+]?\d+(?:[.,]\d+)?(?:\s*/\s*[-+]?\d+(?:[.,]\d+)?)?\s*%?', text.replace(chr(92), ''))
    values = []
    for token in tokens:
        percent = '%' in token
        token = token.replace('%', '').replace(',', '.').strip()
        parts = [float(x) for x in token.split('/')]
        if len(parts) == 2 and parts[1] == 0:
            continue
        value = parts[0] / parts[1] if len(parts) == 2 else parts[0]
        values.append(value / 100 if percent else value)
    return values

def grade(row):
    if not row.received or row['тип'] not in ('A', 'B', 'E'):
        return pd.Series([np.nan, '', 'нет ответа или нужна ручная оценка'])
    tail = re.split(r'ответ\s*:', row.answer, flags=re.I)[-1].replace('{,}', '.')
    if row['тип'] == 'B':
        match = re.search(r'\b(неверно|верно|false|true)\b', row.answer, re.I)
        value = {'true': 'верно', 'false': 'неверно'}.get(match[0].lower(), match[0].lower()) if match else ''
        return pd.Series([float(value == row['эталон_метка']), value, 'метка; обоснование отдельно'])
    values = numbers(tail)
    numeric_gold, label_gold = row['эталон_число'], row['эталон_метка']
    if row['тип'] == 'E' or not numeric_gold:
        valid = len(values) == 1
        correct = valid and values[0] == float(label_gold)
    else:
        tolerance = float(row['допуск_число'] or ('0' if 'точный' in row['допуск'] else '0.01'))
        valid = len(values) == 2 if label_gold else bool(values)
        estimate = values[0] if label_gold and valid else (values[-1] if values else np.nan)
        correct = valid and np.isclose(estimate, float(numeric_gold), rtol=tolerance, atol=1e-12)
        if label_gold and valid:
            correct = correct and values[1] == float(label_gold)
    return pd.Series([float(correct), json.dumps(values), 'проверено автоматически' if valid else 'формат не разобран: проверить извлечение'])


df[['correct', 'extracted', 'grading_note']] = df.apply(grade, axis=1)
df['refusal'] = df.answer.str.contains(r'не могу ответить|нельзя определить|невозможно определить|недостаточно данных|данных недостаточно', case=False, regex=True) | (df.received & df.answer.eq(''))

## Ручные оценки
Передай проверяющим только `reviewer_1.csv` и `reviewer_2.csv`: модели скрыты. `score`: 0/1/2 для B/C/F; для D — `pass_at_1`, `result_correct`, `method_error` (0/1). Код D лежит в `review_code/<case_id>.py` без изменений. При разногласии нужна переоценка; κ < 0,6 требует уточнить рубрику.
`reviewer_type=ai_preliminary` означает предварительную разметку ИИ. После своей проверки участник указывает `human`. Каппа считается только по двум человеческим оценкам; предварительные оценки используются в сводке с отдельной пометкой.

In [ ]:
df['case_id'] = df.apply(lambda r: hashlib.sha256((r.id + r.model + r.answer).encode()).hexdigest()[:12], axis=1)
cases = df.loc[df.received & df['тип'].isin(['B', 'C', 'D', 'F'])]
columns = ['score', 'pass_at_1', 'result_correct', 'method_error']
for r in cases.loc[cases['тип'].eq('D')].itertuples():
    (OUT / 'review_code').mkdir(exist_ok=True)
    (OUT / 'review_code' / f'{r.case_id}.py').write_text(r.answer)
save(cases[['case_id', 'id', 'model']], 'review_key.csv')
for i in [1, 2]:
    path = OUT / f'reviewer_{i}.csv'
    previous = pd.read_csv(path).fillna('') if path.exists() else pd.DataFrame(columns=['case_id', *columns, 'comment'])
    for field, default in [('reviewer_type', 'human'), ('refusal', '')]:
        if field not in previous: previous[field] = default
    form = cases[['case_id', 'id', 'тип', 'текст_ru', 'answer', 'эталон', 'рубрика']].merge(previous[['case_id', *columns, 'comment', 'reviewer_type', 'refusal']], on='case_id', how='left').fillna('')
    form['reviewer_type'] = form.reviewer_type.replace('', 'human')
    save(form, path.name)
    marks = form[['case_id', *columns, 'reviewer_type', 'refusal']].copy()
    marks[columns + ['refusal']] = marks[columns + ['refusal']].apply(pd.to_numeric, errors='coerce')
    df = df.merge(marks.rename(columns={c: f'{c}_{i}' for c in columns + ['reviewer_type', 'refusal']}), on='case_id', how='left')
human_pair = df.reviewer_type_1.eq('human') & df.reviewer_type_2.eq('human')
for c in columns:
    df[c] = df[f'{c}_1'].where(human_pair & df[f'{c}_1'].eq(df[f'{c}_2']))
    preliminary = df.reviewer_type_1.eq('ai_preliminary') & df[c].isna()
    df.loc[preliminary, c] = df.loc[preliminary, f'{c}_1']
manual = df['тип'].isin(['C', 'F']) & df.score.notna()
df.loc[manual, 'correct'] = df.loc[manual, 'score'].eq(2).astype(float)
done = df['тип'].eq('D') & df[['pass_at_1', 'result_correct', 'method_error']].notna().all(axis=1)
df.loc[done, 'correct'] = (df.loc[done, 'pass_at_1'].eq(1) & df.loc[done, 'result_correct'].eq(1) & df.loc[done, 'method_error'].eq(0)).astype(float)
df['assessment_source'] = np.where(df['тип'].isin(['A','B','E']), 'automatic', 'pending')
df.loc[(manual | done) & df.reviewer_type_1.eq('ai_preliminary'), 'assessment_source'] = 'ai_preliminary'
df.loc[(manual | done) & human_pair, 'assessment_source'] = 'human_agreement'
df['refusal'] = df.refusal_1.where(df.refusal_1.notna(), df.refusal).astype(bool)
kappa_rows = []
for kind, field in [('C', 'score'), ('D', 'pass_at_1'), ('D', 'result_correct'), ('D', 'method_error')]:
    pair = df.loc[df['тип'].eq(kind) & human_pair, [f'{field}_1', f'{field}_2']].dropna()
    kappa = cohen_kappa_score(pair.iloc[:, 0], pair.iloc[:, 1]) if len(pair) > 1 and np.unique(pair).size > 1 else np.nan
    kappa_rows.append({'тип': kind, 'оценка': field, 'n': len(pair), 'kappa': kappa})
kappa = pd.DataFrame(kappa_rows)
code_metrics = df.loc[df['тип'].eq('D')].groupby('model')[['pass_at_1', 'result_correct', 'method_error']].mean()
display(kappa, code_metrics)


## Метрики и статистика
Accuracy считается по проверенным ответам; ошибки API показаны отдельно. 95% CI: 1000 bootstrap-выборок, seed=42. При всех верных ответах CI вырождается — малая выборка не доказывает надёжность. Макнемар использует только общие проверенные задачи.

In [ ]:
summary = []
for model in MODELS:
    group = df.loc[df.model.eq(model)]
    values = group.correct.dropna().to_numpy()
    ci = np.quantile(np.random.default_rng(42).choice(values, (1000, len(values))).mean(axis=1), [0.025, 0.975]) if len(values) else [np.nan, np.nan]
    received = group.loc[group.received]
    summary.append({'model': model, 'answered': len(received), 'checked': len(values),
                    'correct': int(values.sum()), 'ai_preliminary': group.assessment_source.eq('ai_preliminary').sum(), 'accuracy': values.mean() if len(values) else np.nan,
                    'ci_low': ci[0], 'ci_high': ci[1], 'api_errors': group.error.ne('').sum(),
                    'refusal_rate': received.refusal.mean(), 'seconds': received.seconds.mean(),
                    'tokens': received.tokens.mean(), 'peak_gpu_gib': received.peak_gpu_gib.max(), 'rubric': received.score.mean()})
summary = pd.DataFrame(summary)
by_type = df.groupby(['model', 'тип']).correct.agg(['mean', 'count']).reset_index()
paired = df.pivot(index='id', columns='model', values='correct')
comparisons = []
for a, b in itertools.combinations(MODELS, 2):
    pair = paired[[a, b]].dropna()
    table = [[int((pair[a].eq(x) & pair[b].eq(y)).sum()) for y in [1, 0]] for x in [1, 0]]
    comparisons.append({'model_a': a, 'model_b': b, 'n': len(pair), 'p_value': mcnemar(table, exact=True).pvalue if len(pair) else np.nan})
comparisons = pd.DataFrame(comparisons)
display(summary.round(4), comparisons)

## Стоимость
Для GPT справочный стандартный тариф — $2 / $10 за 1 млн входных/выходных токенов ([OpenAI](https://developers.openai.com/api/docs/models/gpt-6.1-sol), сверено 29.10.2026). Это не выписка посредника LLM Gate. Без раздельных токенов показывается диапазон.
GigaChat-3-Ultra доступна только во freemium: платный тариф не опубликован ([Сбер](https://developers.sber.ru/docs/ru/gigachat/models/gigachat-3-ultra)). Поэтому денежный эквивалент обозначен как N/A, а не заменён ценой другой модели. Для локальных моделей сравниваем время и память GPU.

In [ ]:
fx = {'2026-10-28': 80.5307, '2026-10-29': 80.8664}
tariffs = {'gpt': (2, 10)}
costs = []
for model, (input_price, output_price) in tariffs.items():
    group = df.loc[df.model.eq(model) & df.received]
    rate = group.date.str[:10].map(fx)
    exact = (pd.to_numeric(group.prompt_tokens) * input_price + pd.to_numeric(group.completion_tokens) * output_price) * rate / 1e6
    low = exact.fillna(pd.to_numeric(group.tokens) * min(input_price, output_price) * rate / 1e6)
    high = exact.fillna(pd.to_numeric(group.tokens) * max(input_price, output_price) * rate / 1e6)
    correct = group.correct.sum()
    costs.append({'model': model, 'rub_per_correct_low': low.sum(min_count=len(group)) / correct if correct else np.nan,
                  'rub_per_correct_high': high.sum(min_count=len(group)) / correct if correct else np.nan})
costs.append({'model': 'gigachat', 'rub_per_correct_low': np.nan, 'rub_per_correct_high': np.nan})
costs = pd.DataFrame(costs)
display(costs)

## Обязательные графики

In [ ]:
heat = by_type.pivot(index='model', columns='тип', values='mean').reindex(index=MODELS, columns=list('ABCDEF'))
counts = by_type.pivot(index='model', columns='тип', values='count').reindex(index=MODELS, columns=list('ABCDEF'))
fig, ax = plt.subplots(figsize=(9, 3), layout='constrained')
cmap = plt.colormaps['Blues'].copy(); cmap.set_bad('#eeeeee')
image = ax.imshow(np.ma.masked_invalid(heat.to_numpy()), cmap=cmap, vmin=0, vmax=1, aspect='auto')
ax.set(xticks=range(6), xticklabels=list('ABCDEF'), yticks=range(len(MODELS)), yticklabels=MODELS, title='Accuracy: модель × тип задачи')
for i, model in enumerate(MODELS):
    for j, kind in enumerate('ABCDEF'):
        value = heat.loc[model, kind]
        ax.text(j, i, f'{value:.0%}\nn={counts.loc[model, kind]}' if pd.notna(value) else '—', ha='center', va='center', color='white' if pd.notna(value) and value > 0.6 else 'black')
fig.colorbar(image, ax=ax); fig.savefig(OUT / 'accuracy_by_type.png', dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
for i, r in summary.iterrows():
    if pd.notna(r.accuracy):
        ax.errorbar(r.accuracy, i, xerr=[[max(0, r.accuracy-r.ci_low)], [max(0, r.ci_high-r.accuracy)]], fmt='o', capsize=4)
        ax.annotate(f'n={r.checked}', (r.accuracy, i), xytext=(-30, 8), textcoords='offset points')
ax.set(yticks=range(len(MODELS)), yticklabels=MODELS, xlim=(-0.03, 1.08), ylim=(-0.5, len(MODELS)-0.5), xlabel='Accuracy и 95% CI')
fig.savefig(OUT / 'accuracy_ci.png', dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 3), layout='constrained')
for i, r in costs.dropna().iterrows():
    accuracy = summary.set_index('model').loc[r.model, 'accuracy']
    center = (r.rub_per_correct_low + r.rub_per_correct_high) / 2
    ax.errorbar(center, accuracy, xerr=(r.rub_per_correct_high-r.rub_per_correct_low)/2, fmt='o', capsize=4)
    ax.annotate(r.model, (center, accuracy), xytext=(0, 12+14*i), textcoords='offset points', ha='center')
ax.set(xlabel='Рублей за верный ответ: диапазон', ylabel='Accuracy', ylim=(0, 1.3))
ax.text(.02, .03, 'GigaChat Ultra: платный тариф N/A; local/T-lite: время и GPU-память', transform=ax.transAxes, fontsize=8)
fig.savefig(OUT / 'cost_accuracy.png', dpi=150); plt.show()

## Тип E: F1 редкого класса и balanced accuracy
Тестовые 150 строк и непересекающиеся support-наборы 5/20 уже сохранены в `classification_test.csv` и `classification_support.csv`. Метки теста нельзя передавать LLM. Сравнение с LLM возможно после её прогона тех же строк в режимах 0/5/20-shot.

In [ ]:
%run analyze_experiments.py
e_metrics = pd.read_csv(OUT / 'classification_with_ci.csv')
display(e_metrics.round(4))


## Результаты
Ошибки решений сохраняются отдельно от ошибок API. Дополнительные прогоны и их полнота представлены в таблицах H3/H5/H12 и classification_coverage. Протокол зафиксирован после пилота; независимая человеческая проверка ещё требуется. Итоговые выводы пишет автор работы.

In [ ]:
wrong = df.loc[df.correct.eq(0) | df.score.lt(2) | df.method_error.eq(1), ['id', 'model', 'тип', 'answer', 'эталон']]
tables = {'summary': summary, 'by_type': by_type, 'mcnemar': comparisons, 'costs': costs, 'kappa': kappa,
          'code_metrics': code_metrics.reset_index(), 'classification_metrics': e_metrics,
          'wrong_answers': wrong, 'api_errors': df.loc[df.error.ne(''), ['id','model','error']],
          'evaluated_answers': df[['id','model','тип','answer','correct','extracted','grading_note','assessment_source']]}
for name, table in tables.items():
    save(table, name + '.csv')
html = '<html lang="ru"><meta charset="utf-8"><style>body{font:15px system-ui;margin:30px}table{border-collapse:collapse}td,th{border:1px solid #ddd;padding:8px}img{display:block;max-width:100%;width:900px}</style><h1>ЛР 3 — результаты</h1><p>Сводка включает предварительные оценки ИИ. Два независимых человеческих проверяющих ещё не завершили C/D; каппа по ИИ не вычисляется. Accuracy относится только к проверенным ответам; ошибки API считаются отдельно. Стоимость для старых записей — диапазон, без затрат запросов, для которых нет usage. Доверительные интервалы могут вырождаться на малой выборке. У GigaChat Ultra нет опубликованного платного тарифа; денежный эквивалент N/A. Для открытых моделей приведены время и GPU-память. Полнота новых экспериментов показана отдельно; результаты с разметкой ИИ предварительные.</p>'
for title, table in [('Метрики', summary), ('Макнемар', comparisons), ('Стоимость', costs), ('Каппа', kappa), ('Код D', code_metrics.reset_index()), ('Классификация E', e_metrics), ('Ошибки решений', wrong)]:
    html += '<h2>' + title + '</h2>' + table.to_html(index=False, na_rep='—', float_format=lambda x: f'{x:.4f}')
html += ''.join(f'<img src="{name}.png">' for name in ['accuracy_by_type','accuracy_ci','cost_accuracy'])
for title, filename in [('Полнота классификации', 'classification_coverage.csv'), ('H3: недостаток данных', 'hypothesis_H3.csv'), ('H5: разрыв A/B', 'hypothesis_H5.csv'), ('H12: разность с LogReg', 'hypothesis_H12.csv')]:
    path = OUT / filename
    if path.exists():
        table = pd.read_csv(path)
        html += '<h2>' + title + '</h2>' + (table.to_html(index=False, na_rep='—') if len(table) else '<p>Полных данных пока нет.</p>')
html += ''.join(f'<img src="{name}.png">' for name in ['classification_few_shot', 'hypothesis_controls'] if (OUT / f'{name}.png').exists())
(OUT / 'report.html').write_text(html + '</html>', encoding='utf-8')
print(f'Оценено: {df.correct.notna().sum()} из {len(df)}; из них предварительно ИИ: {df.assessment_source.eq("ai_preliminary").sum()}. Ошибок по текущим критериям: {len(wrong)}.')